In [ ]:
from BUSBRA_Medical import BUSBRADataModule
from classifiers import ClassificationModel
import pytorch_lightning as pl
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint

## Setando seeds para reprodutibilidade

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    # Set the seed for Python's built-in random library
    random.seed(seed)
    
    # Set the seed for NumPy
    np.random.seed(seed)
    
    # Set the seed for PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if using multi-GPU
    
    # For performance reasons, this setting should only be enabled for true reproducibility.
    # It can lead to slower training in some cases.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Set the seed for PyTorch Lightning
    pl.seed_everything(seed, workers=True)

# Call the function to set seeds
set_seed(42)

## Carregando CSV do dataset

In [ ]:
import pandas as pd

merged_df = df = pd.read_csv('../busbra_medical_error.csv')
merged_df.head()

## Teste do modelo

In [ ]:
from sklearn.model_selection import KFold
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from matplotlib import pyplot as plt
import json

Kfold = [1,2,3,4,5]
fold_results = []
fold_cm = []
corr_image = []
true_list = []
model_name = 'swint'
moment = '-27-05-25-15-02-19'

important_images = {
    'med_err_mod_corr': [],
    'med_corr_mod_err': [],
    'med_corr_mod_corr': [],
    'med_err_mod_err': [],
    
}

for i in Kfold:
    pred2 = []
    pred3 = []
    pred4 = []
    pred5 = []

    y2 = []
    y3 = []
    y4 = []
    y5 = []

    model_tp_count = 0
    model_tn_count = 0
    doctor_tp_count = 0
    doctor_tn_count = 0
    
    print(f'####### Fold {i} #######')
    column = 'valid_' + str(i)
    test_df = merged_df[merged_df['kFold'] == i]  
    test_df.reset_index(drop=False,inplace=True)
    dm = BUSBRADataModule(Kfold=True, test=test_df)    

    ckpt_path = 'weights/'+model_name +'/BUSBRA-'+model_name +'-fold-'+ str(i) + moment +'.ckpt'
    
    trainer = pl.Trainer(
        accelerator="gpu", 
        devices=1, 
        )

    model = ClassificationModel.load_from_checkpoint(ckpt_path)
    results = trainer.test(model=model, datamodule=dm)
    fold_results.append(results)

    preds = trainer.predict(model=model, datamodule=dm)
    predsm = torch.cat([x for x in preds]).argmax(dim=-1)
    
    for pred, y, birad, name, medical_error in zip(predsm, test_df['Pathology'].values,test_df['BIRADS'].values, test_df['ID'].values, test_df['Medical error'].values):
        y = 1 if y=='malignant' else 0   

        if (birad==2 or birad==3) and y==0:
            doctor_tn_count += 1
            if pred==y:
                model_tn_count += 1   
        elif (birad==4 or birad==5) and y==1:
            doctor_tp_count += 1
            if pred==y:
                model_tp_count += 1
        
        if birad == 2:
            pred2.append(int(pred))
            y2.append(y)
        elif birad == 3:
            pred3.append(int(pred))
            y3.append(y)
        elif birad == 4:
            pred4.append(int(pred))
            y4.append(y)
        elif birad == 5:
            pred5.append(int(pred))
            y5.append(y)
        
        # medico errou e modelo acertou
        if medical_error!='none' and pred==y:
            important_images['med_err_mod_corr'].append(name)
        # medico acertou e modelo errou
        if medical_error=='none' and pred!=y:
            important_images['med_corr_mod_err'].append(name)
        # medico acertou e modelo acertou
        if medical_error=='none' and pred==y:
            important_images['med_corr_mod_corr'].append(name)
        # medico errou e modelo errou
        if medical_error!='none' and pred!=y:
            important_images['med_err_mod_err'].append(name)
            
    birads_dict = {'2':confusion_matrix(y2, pred2, labels=[0, 1]),
                   '3':confusion_matrix(y3, pred3, labels=[0, 1]),
                   '4':confusion_matrix(y4, pred4, labels=[0, 1]),
                   '5':confusion_matrix(y5, pred5, labels=[0, 1])
                  }

    true_list.append([model_tp_count, doctor_tp_count, model_tn_count, doctor_tn_count])
    fold_cm.append(birads_dict)

## BUSBRA Métricas gerais

In [ ]:
# geral
import numpy as np

ms_fn = 0
ms_fp = 0
for i in range(len(fold_cm)):
    ms_fn += fold_cm[i]['2'][1][1] + fold_cm[i]['3'][1][1]
    ms_fp += fold_cm[i]['4'][0][0] + fold_cm[i]['5'][0][0]

m_tp = 0
d_tp = 0
m_tn = 0
d_tn = 0
for i in range(len(true_list)):
    m_tp += true_list[i][0]
    d_tp += true_list[i][1]
    m_tn += true_list[i][2]
    d_tn += true_list[i][3]

sens = [fold_results[i][0]['test_sens'] for i in range(0,5)]
spec = [fold_results[i][0]['test_spec'] for i in range(0,5)]
accuracy = [fold_results[i][0]['test_acc'] for i in range(0,5)]
f1_score = [fold_results[i][0]['test_f1_score'] for i in range(0,5)]

all_results = ''

print(f'======= {model_name} BUSBRA =======\n')
# print para o LaTeX
results_latex = f"BUSBRA Geral:Swin & {round(np.mean(accuracy)*100,2)} \u00B1 {round(np.std(accuracy)*100,2)} & {round(np.mean(sens)*100,2)} \u00B1 {round(np.std(sens)*100,2)} & {round(np.mean(spec)*100,2)} \u00B1 {round(np.std(spec)*100,2)} & {round(np.mean(f1_score)*100,2)} \u00B1 {round(np.std(f1_score)*100,2)} & {round(100*ms_fn / 41, 2)}%({ms_fn}/41) & {round(100*ms_fp / 284, 2)}%({ms_fp}/284) & {round(100*m_tn / d_tn, 2)}%({m_tn}/{d_tn}) & {round(100*m_tp / d_tp, 2)}\\%({m_tp}/{d_tp})\n"
print('Model            | Accuracy     | Sensitivity | Specificity  | F1-Score')
print('| M/S-FN       | M/S-FP | M/S-TN | M/S-TP')
print(results_latex)

all_results = results_latex 

## BUSBRA BI-RADS 3 e 4

In [ ]:
# calculo para 3 e 4
accuracy = []
sens = []
spec = []
f1 = []

m_fp = 0
s_fp = 0
m_fn = 0
s_fn = 0

m_tp = 0
s_tp = 0
m_tn = 0
s_tn = 0

for i in range(len(fold_cm)):
    # metricas tradicionais
    cm_34 = fold_cm[i]['3'] + fold_cm[i]['4']
    tn, fp, fn, tp = cm_34.ravel()
    
    accuracy.append((tp + tn) / (tp + tn + fp + fn))
    sens.append(tp / (tp + fn))
    spec.append(tn / (tn + fp))
    sens_aux = tp / (tp + fn)
    prec = tp / (tp + fp)
    f1.append(2 * (prec * sens_aux) / (prec + sens_aux))

    # metricas novas

    # MS-FN -> quantas o modelo acertou das que sao malignas e o medico classificou como benignas
    m_fn += fold_cm[i]['3'][1][1]
    s_fn += (fold_cm[i]['3'][1][1] + fold_cm[i]['3'][1][0])
    
    # MS-FP -> quantas o modelo acertou das que sao benignas e o medico classificou como malignas
    m_fp += fold_cm[i]['4'][0][0]
    s_fp += (fold_cm[i]['4'][0][0] + fold_cm[i]['4'][0][1])
    
    # MS-TN -> quantas o modelo acertou das que sao benignas e o medico classificou como benignas
    m_tn += fold_cm[i]['3'][0][0]
    s_tn += (fold_cm[i]['3'][0][0] + fold_cm[i]['3'][0][1])
    
    # MS-TP -> quantas o modelo acertou das que sao malignas e o medico classificou como malignas
    m_tp += fold_cm[i]['4'][1][1]
    s_tp += (fold_cm[i]['4'][1][1] + fold_cm[i]['4'][1][0])
    
print(f'======= {model_name} BUSBRA for BI-RADS 3 and 4 =======\n')
# print para o LaTeX
results_latex = f"BUSBRA 3 e 4:Swin & {round(np.mean(accuracy)*100,2)} \u00B1 {round(np.std(accuracy)*100,2)} & {round(np.mean(sens)*100,2)} \u00B1 {round(np.std(sens)*100,2)} & {round(np.mean(spec)*100,2)} \u00B1 {round(np.std(spec)*100,2)} & {round(np.mean(f1)*100,2)} \u00B1 {round(np.std(f1)*100,2)} & {round(100*m_fn / s_fn, 2)}%({m_fn}/{s_fn}) & {round(100*m_fp / s_fp, 2)}%({m_fp}/{s_fp}) & {round(100*m_tn / s_tn, 2)}%({m_tn}/{s_tn}) & {round(100*m_tp / s_tp, 2)}\\%({m_tp}/{s_tp}) \n"
print('Model            | Accuracy     | Sensitivity | Specificity  | F1-Score')
print('| M/S-FN       | M/S-FP | M/S-TN | M/S-TP')
print(results_latex)

all_results = all_results + results_latex 

## BUSBRA BI-RADS 2 e 5

In [ ]:
# calculo para 2 e 5
accuracy = []
sens = []
spec = []
f1 = []

m_fp = 0
s_fp = 0
m_fn = 0
s_fn = 0

m_tp = 0
s_tp = 0
m_tn = 0
s_tn = 0

for i in range(len(fold_cm)):
    # metricas tradicionais
    cm_25 = fold_cm[i]['2'] + fold_cm[i]['5']
    tn, fp, fn, tp = cm_25.ravel()
    
    accuracy.append((tp + tn) / (tp + tn + fp + fn))
    sens.append(tp / (tp + fn))
    spec.append(tn / (tn + fp))
    sens_aux = tp / (tp + fn)
    prec = tp / (tp + fp)
    f1.append(2 * (prec * sens_aux) / (prec + sens_aux))

    # metricas novas

    # MS-FN -> quantas o modelo acertou das que sao malignas e o medico classificou como benignas
    m_fn += fold_cm[i]['2'][1][1]
    s_fn += (fold_cm[i]['2'][1][1] + fold_cm[i]['2'][1][0])
    
    # MS-FP -> quantas o modelo acertou das que sao benignas e o medico classificou como malignas
    m_fp += fold_cm[i]['5'][0][0]
    s_fp += (fold_cm[i]['5'][0][0] + fold_cm[i]['5'][0][1])
    
    # MS-TN -> quantas o modelo acertou das que sao benignas e o medico classificou como benignas
    m_tn += fold_cm[i]['2'][0][0]
    s_tn += (fold_cm[i]['2'][0][0] + fold_cm[i]['2'][0][1])
    
    # MS-TP -> quantas o modelo acertou das que sao malignas e o medico classificou como malignas
    m_tp += fold_cm[i]['5'][1][1]
    s_tp += (fold_cm[i]['5'][1][1] + fold_cm[i]['5'][1][0])
    
print(f'======= {model_name} BUSBRA for BI-RADS 2 and 5 =======\n')
# print para o LaTeX
results_latex = f"BUSBRA 2 e 5:Swin AdaSampling & {round(np.mean(accuracy)*100,2)} \u00B1 {round(np.std(accuracy)*100,2)} & {round(np.mean(sens)*100,2)} \u00B1 {round(np.std(sens)*100,2)} & {round(np.mean(spec)*100,2)} \u00B1 {round(np.std(spec)*100,2)} & {round(np.mean(f1)*100,2)} \u00B1 {round(np.std(f1)*100,2)} & {round(100*m_fn / s_fn, 2)}%({m_fn}/{s_fn}) & {round(100*m_fp / s_fp, 2)}%({m_fp}/{s_fp}) & {round(100*m_tn / s_tn, 2)}%({m_tn}/{s_tn}) & {round(100*m_tp / s_tp, 2)}\\%({m_tp}/{s_tp})\n"
print('Model            | Accuracy     | Sensitivity | Specificity  | F1-Score')
print('| M/S-FN       | M/S-FP | M/S-TN | M/S-TP')
print(results_latex)
all_results = all_results + results_latex

## Validação Externa 

### BUSI

In [ ]:
from BUSI_Medical_EV import BUSIDataModule

df = pd.read_csv('../Dataset_BUSI_with_GT/BUSI_corrigido.csv')

In [ ]:
dm = BUSIDataModule(df, dataset_name='busi')
dm.imshow_test()

In [ ]:
Kfold = [1,2,3,4,5]

busi_fold_results = []

for i in Kfold:
    print(f'####### Fold {i} #######')
    column = 'valid_' + str(i)
    test_df = merged_df[merged_df['kFold'] == i]  
    dm = BUSIDataModule(df, dataset_name='busi')    

    ckpt_path = 'weights/'+model_name+'/BUSBRA-'+model_name+'-fold-'+ str(i) + moment + '.ckpt'
    
    trainer = pl.Trainer(
        accelerator="gpu", 
        devices=1, 
        )

    model = ClassificationModel.load_from_checkpoint(ckpt_path)
    results = trainer.test(model=model, datamodule=dm)
    busi_fold_results.append(results)

In [ ]:
sens = [busi_fold_results[i][0]['test_sens'] for i in range(0,5)]
spec = [busi_fold_results[i][0]['test_spec'] for i in range(0,5)]
accuracy = [busi_fold_results[i][0]['test_acc'] for i in range(0,5)]
f1_score = [busi_fold_results[i][0]['test_f1_score'] for i in range(0,5)]

print(f'======================== {model_name} BUSI ========================\n')
# print para o LaTeX
results_latex = f"BUSI:Swin & {round(np.mean(accuracy)*100,2)} \u00B1 {round(np.std(accuracy)*100,2)} & {round(np.mean(sens)*100,2)} \u00B1 {round(np.std(sens)*100,2)} & {round(np.mean(spec)*100,2)} \u00B1 {round(np.std(spec)*100,2)} & {round(np.mean(f1_score)*100,2)} \u00B1 {round(np.std(f1_score)*100,2)}\n"
print('Model            | Accuracy     | Sensitivity | Specificity  | F1-Score')
print(results_latex)
all_results = all_results + results_latex 

### BrEaST

In [ ]:
from BUSI_Medical_EV import BUSIDataModule
import pandas as pd

df = pd.read_csv('../BrEaST-Lesions_USG-images_and_masks-Dec-15-2023/BrEaSt_dataset.csv')
dm = BUSIDataModule(df, dataset_name='breast')
dm.imshow_test()

In [ ]:
Kfold = [1,2,3,4,5]

breast_fold_results = []

for i in Kfold:
    print(f'####### Fold {i} #######')
    column = 'valid_' + str(i)
    dm = BUSIDataModule(df, dataset_name='breast')    

    ckpt_path = 'weights/'+model_name+'/BUSBRA-'+model_name+'-fold-'+ str(i) + moment + '.ckpt'
    
    trainer = pl.Trainer(
        accelerator="gpu", 
        devices=1, 
        )

    model = ClassificationModel.load_from_checkpoint(ckpt_path)
    results = trainer.test(model=model, datamodule=dm)
    breast_fold_results.append(results)

In [ ]:
import numpy as np

sens = [breast_fold_results[i][0]['test_sens'] for i in range(0,5)]
spec = [breast_fold_results[i][0]['test_spec'] for i in range(0,5)]
accuracy = [breast_fold_results[i][0]['test_acc'] for i in range(0,5)]
f1_score = [breast_fold_results[i][0]['test_f1_score'] for i in range(0,5)]

print(f'======================== {model_name} BrEaSt ========================\n')
# print para o LaTeX
results_latex = f"BrEaSt:Swin & {round(np.mean(accuracy)*100,2)} \u00B1 {round(np.std(accuracy)*100,2)} & {round(np.mean(sens)*100,2)} \u00B1 {round(np.std(sens)*100,2)} & {round(np.mean(spec)*100,2)} \u00B1 {round(np.std(spec)*100,2)} & {round(np.mean(f1_score)*100,2)} \u00B1 {round(np.std(f1_score)*100,2)}\n"
print('Model            | Accuracy     | Sensitivity | Specificity  | F1-Score')
print(results_latex)
all_results = all_results + results_latex 

In [ ]:
# Salvar no arquivo txt
with open("results/"+model_name+"/"+model_name+"_results" + moment + ".txt", "w", encoding="utf-8") as file:
    file.write(all_results + "\n\n")  # Escreve a string no topo com uma linha em branco
    json.dump(fold_results, file, indent=4)  # Salva a lista formatada em JSON
with open("results/"+model_name+"/"+model_name+"_important_images-" + moment + ".txt", "w", encoding="utf-8") as file:
    json.dump(important_images, file, indent=4)  # Salva a lista formatada em JSON